In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.tsa.holtwinters import ExponentialSmoothing

df = pd.read_csv("data/processed/retail_inventory_clean.csv", parse_dates=["date"])

In [2]:
df["stockout_day"] = df["inventory_level"] == 0
stockout_rate = df["stockout_day"].mean()

print(f"Stockout rate across all store-product-days: {stockout_rate*100:.1f}%")
print(f"Total recorded revenue: ${df['revenue'].sum():,.0f}")

Stockout rate across all store-product-days: 0.5%
Total recorded revenue: $412,590,590


In [3]:
weekly = df.set_index("date").resample("W")["units_sold"].sum()
weekly_bundled = df.set_index("date").resample("W")["demand"].sum()  # the dataset's own bundled forecast column

if df["date"].max() < weekly.index[-1]:
    weekly, weekly_bundled = weekly.iloc[:-1], weekly_bundled.iloc[:-1]  # drop a trailing partial week

horizon = 12
train, test = weekly.iloc[:-horizon], weekly.iloc[-horizon:]

hw = ExponentialSmoothing(train, trend="add").fit()
forecast = hw.forecast(horizon)
naive = pd.Series(train.iloc[-horizon:].values, index=test.index)  # last observed 12 weeks, repeated forward
bundled = weekly_bundled.reindex(test.index)

mape = lambda actual, pred: (abs(actual - pred) / actual).mean() * 100
print(f"Holt-Winters MAPE:               {mape(test, forecast):.1f}%")
print(f"Naive (repeat last period) MAPE: {mape(test, naive):.1f}%")
print(f"Dataset's own Demand field MAPE: {mape(test, bundled):.1f}%")

Holt-Winters MAPE:               9.8%
Naive (repeat last period) MAPE: 9.0%
Dataset's own Demand field MAPE: 17.0%


In [4]:
LEAD_TIME_DAYS = 7    # stated planning assumption: how long a restock order takes to arrive -- not in the data
SERVICE_Z = 1.65      # z-score for a 95% service level -- tested against other levels in Cell 6

policy = df.groupby(["store_id", "product_id"]).agg(
    mean_demand=("units_sold", "mean"), std_demand=("units_sold", "std"),
    avg_price=("price", "mean"), avg_inventory=("inventory_level", "mean"),
    stockout_days=("stockout_day", "sum"), total_days=("date", "count")).fillna(0)

policy["safety_stock"] = SERVICE_Z * policy.std_demand * np.sqrt(LEAD_TIME_DAYS)
policy["reorder_point"] = policy.mean_demand * LEAD_TIME_DAYS + policy.safety_stock
policy["stockout_rate"] = policy.stockout_days / policy.total_days
policy["overstocked"] = policy.avg_inventory > policy.reorder_point * 2

policy[["mean_demand", "reorder_point", "avg_inventory", "stockout_rate", "overstocked"]].describe()

,mean_demand,reorder_point,avg_inventory,stockout_rate
count,100.000000,100.000000,100.000000,100.000000
mean,88.827316,799.714539,301.062842,0.005342
std,15.126384,132.160230,144.424523,0.005018
min,61.798684,550.252751,102.119737,0.000000
25%,77.103947,713.862424,151.478289,0.000000
50%,93.262500,818.137015,278.533553,0.005263
75%,102.893092,927.397760,398.366118,0.009211
max,108.843421,984.617283,589.967105,0.018421


In [5]:
HOLDING_COST_RATE = 0.20   # annual carrying-cost planning figure, typical for retail -- tested at other rates in Cell 6

policy["lost_revenue"] = policy.stockout_days * policy.mean_demand * policy.avg_price
policy["excess_units"] = (policy.avg_inventory - policy.reorder_point).clip(lower=0)
policy["holding_cost"] = policy.excess_units * policy.avg_price * HOLDING_COST_RATE

print(f"Estimated lost revenue from stockouts:    ${policy.lost_revenue.sum():,.0f}")
print(f"Estimated holding cost from excess stock: ${policy.holding_cost.sum():,.0f}")

Estimated lost revenue from stockouts:    $2,381,024
Estimated holding cost from excess stock: $0


In [6]:
scenarios = [("Current (actual)", None), ("80% service level", 1.28),
             ("95% service level", 1.65), ("99% service level", 2.33)]
sim_rows = []
for label, z in scenarios:
    if z is None:
        lost, hold = policy.lost_revenue.sum(), policy.holding_cost.sum()
    else:
        target_stockout_rate = 1 - norm.cdf(z)
        safety = z * policy.std_demand * np.sqrt(LEAD_TIME_DAYS)
        lost = (target_stockout_rate * policy.total_days * policy.mean_demand * policy.avg_price).sum()
        hold = (safety * policy.avg_price * HOLDING_COST_RATE).sum()
    total = lost + hold
    sim_rows.append({"policy": label, "lost_revenue": lost, "holding_cost": hold, "total_cost": total})
    print(f"{label}: lost revenue ${lost:,.0f}, holding cost ${hold:,.0f}, total ${total:,.0f}")

sim_df = pd.DataFrame(sim_rows)

Current (actual): lost revenue $2,381,024, holding cost $0, total $2,381,024
80% service level: lost revenue $43,814,225, holding cost $178,070, total $43,992,295
95% service level: lost revenue $21,616,620, holding cost $229,544, total $21,846,164
99% service level: lost revenue $4,327,161, holding cost $324,143, total $4,651,305


In [7]:
weekly_export = pd.DataFrame({"week": weekly.index, "actual": weekly.values})
weekly_export.loc[weekly_export.index[-horizon:], "forecast"] = forecast.values
weekly_export.loc[weekly_export.index[-horizon:], "naive"] = naive.values
weekly_export.loc[weekly_export.index[-horizon:], "bundled_demand"] = bundled.values
weekly_export.to_csv("data/processed/demand_forecast.csv", index=False)

category_policy = df.groupby("category").agg(
    stockout_rate=("stockout_day", "mean"), revenue=("revenue", "sum")).reset_index()
category_policy["stockout_rate"] *= 100
category_policy.to_csv("data/processed/category_policy.csv", index=False)

policy.reset_index()[["store_id", "product_id", "stockout_rate", "overstocked", "lost_revenue", "holding_cost"]].to_csv(
    "data/processed/policy_detail.csv", index=False)

sim_df.to_csv("data/processed/policy_simulation.csv", index=False)